In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

In [2]:
DATA_PATH = Path("../data/raw/RML2016.10a/RML2016.10a_dict.pkl")

print(DATA_PATH)
print("Existe:", DATA_PATH.exists())

..\data\raw\RML2016.10a\RML2016.10a_dict.pkl
Existe: True


In [3]:
with open(DATA_PATH, "rb") as f:
    radioml = pickle.load(f, encoding="latin1")

print("Dataset cargado correctamente.")

Dataset cargado correctamente.


In [4]:
print("Tipo:", type(radioml))
print("Número de claves:", len(radioml))

Tipo: <class 'dict'>
Número de claves: 220


In [5]:
keys = list(radioml.keys())

print("Primeras 10 claves:")
for key in keys[:10]:
    print(key)

Primeras 10 claves:
('QPSK', 2)
('PAM4', 8)
('AM-DSB', -4)
('GFSK', 6)
('QAM64', 8)
('AM-SSB', 12)
('8PSK', 8)
('8PSK', 12)
('QAM64', -6)
('QAM16', 2)


In [6]:
modulations = sorted(set(key[0] for key in keys))
snrs = sorted(set(key[1] for key in keys))

print("Modulaciones:")
print(modulations)

print("\nNúmero de modulaciones:", len(modulations))

print("\nSNR disponibles:")
print(snrs)

print("\nNúmero de niveles de SNR:", len(snrs))

Modulaciones:
['8PSK', 'AM-DSB', 'AM-SSB', 'BPSK', 'CPFSK', 'GFSK', 'PAM4', 'QAM16', 'QAM64', 'QPSK', 'WBFM']

Número de modulaciones: 11

SNR disponibles:
[-20, -18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

Número de niveles de SNR: 20


In [7]:
first_key = keys[0]
samples = radioml[first_key]

print("Clave seleccionada:", first_key)
print("Tipo:", type(samples))
print("Shape:", samples.shape)
print("dtype:", samples.dtype)

Clave seleccionada: ('QPSK', 2)
Tipo: <class 'numpy.ndarray'>
Shape: (1000, 2, 128)
dtype: float32


In [8]:
sample = samples[0]

print("Shape de una señal:", sample.shape)

I = sample[0]
Q = sample[1]

print("Shape de I:", I.shape)
print("Shape de Q:", Q.shape)

print("\nPrimeros 5 valores de I:")
print(I[:5])

print("\nPrimeros 5 valores de Q:")
print(Q[:5])

Shape de una señal: (2, 128)
Shape de I: (128,)
Shape de Q: (128,)

Primeros 5 valores de I:
[-0.00590147 -0.00234582 -0.00074506 -0.00534572 -0.00578942]

Primeros 5 valores de Q:
[-0.00779554 -0.00781637 -0.00401967 -0.00511351 -0.00593952]


In [9]:
block_sizes = [samples.shape[0] for samples in radioml.values()]
signal_shapes = [samples.shape[1:] for samples in radioml.values()]
dtypes = [samples.dtype for samples in radioml.values()]

print("Tamaños de bloque distintos:", set(block_sizes))
print("Shapes de señal distintas:", set(signal_shapes))
print("Tipos de dato distintos:", set(dtypes))

print("\nNúmero total de señales:", sum(block_sizes))

Tamaños de bloque distintos: {1000}
Shapes de señal distintas: {(2, 128)}
Tipos de dato distintos: {dtype('float32')}

Número total de señales: 220000


In [10]:
assert len(radioml) == len(modulations) * len(snrs)
assert len(set(signal_shapes)) == 1
assert len(set(dtypes)) == 1

print("✅ Estructura del dataset consistente.")

✅ Estructura del dataset consistente.


In [11]:
records = []
sample_id = 0

for modulation, snr in sorted(radioml.keys()):
    n_samples = radioml[(modulation, snr)].shape[0]

    for array_index in range(n_samples):
        records.append({
            "sample_id": sample_id,
            "modulation": modulation,
            "snr": snr,
            "array_index": array_index
        })

        sample_id += 1

metadata = pd.DataFrame(records)

In [12]:
metadata.head()

,sample_id,modulation,snr,array_index
0,0,8PSK,-20,0
1,1,8PSK,-20,1
2,2,8PSK,-20,2
3,3,8PSK,-20,3
4,4,8PSK,-20,4


In [13]:
print("Shape:", metadata.shape)

print("\nTipos de datos:")
print(metadata.dtypes)

print("\nValores ausentes:")
print(metadata.isna().sum())

metadata.head()

Shape: (220000, 4)

Tipos de datos:
sample_id      int64
modulation       str
snr            int64
array_index    int64
dtype: object

Valores ausentes:
sample_id      0
modulation     0
snr            0
array_index    0
dtype: int64


,sample_id,modulation,snr,array_index
0,0,8PSK,-20,0
1,1,8PSK,-20,1
2,2,8PSK,-20,2
3,3,8PSK,-20,3
4,4,8PSK,-20,4


In [14]:
modulation_counts = metadata["modulation"].value_counts().sort_index()

print(modulation_counts)

modulation
8PSK      20000
AM-DSB    20000
AM-SSB    20000
BPSK      20000
CPFSK     20000
GFSK      20000
PAM4      20000
QAM16     20000
QAM64     20000
QPSK      20000
WBFM      20000
Name: count, dtype: int64


In [15]:
snr_counts = metadata["snr"].value_counts().sort_index()

print(snr_counts)

snr
-20    11000
-18    11000
-16    11000
-14    11000
-12    11000
-10    11000
-8     11000
-6     11000
-4     11000
-2     11000
 0     11000
 2     11000
 4     11000
 6     11000
 8     11000
 10    11000
 12    11000
 14    11000
 16    11000
 18    11000
Name: count, dtype: int64


In [16]:
balance_table = pd.crosstab(
    metadata["modulation"],
    metadata["snr"]
)

balance_table

snr,-20,-18,-16,-14,-12,-10,-8,-6,-4,-2,0,2,4,6,8,10,12,14,16,18
modulation,,,,,,,,,,,,,,,,,,,,
8PSK,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
AM-DSB,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
AM-SSB,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
BPSK,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
CPFSK,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
GFSK,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
PAM4,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
QAM16,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
QAM64,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000


In [17]:
row = metadata.sample(1, random_state=42).iloc[0]

print("Fila seleccionada:")
print(row)

signal_from_metadata = radioml[
    (row["modulation"], int(row["snr"]))
][int(row["array_index"])]

print("\nShape recuperado:", signal_from_metadata.shape)

Fila seleccionada:
sample_id      132386
modulation       PAM4
snr                 4
array_index       386
Name: 132386, dtype: object

Shape recuperado: (2, 128)


In [18]:
from sklearn.model_selection import train_test_split
metadata_split = metadata.copy()

metadata_split["stratum"] = (
    metadata_split["modulation"].astype(str)
    + "_"
    + metadata_split["snr"].astype(str)
)

train_df, temp_df = train_test_split(
    metadata_split,
    test_size=0.30,
    random_state=42,
    stratify=metadata_split["stratum"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["stratum"]
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (154000, 5)
Validation: (33000, 5)
Test: (33000, 5)


In [19]:
train_ids = set(train_df["sample_id"])
validation_ids = set(validation_df["sample_id"])
test_ids = set(test_df["sample_id"])

print(
    "Train ∩ Validation:",
    len(train_ids.intersection(validation_ids))
)

print(
    "Train ∩ Test:",
    len(train_ids.intersection(test_ids))
)

print(
    "Validation ∩ Test:",
    len(validation_ids.intersection(test_ids))
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [20]:
all_split_ids = train_ids | validation_ids | test_ids

print("IDs totales tras el split:", len(all_split_ids))
print("IDs originales:", len(metadata))

assert len(all_split_ids) == len(metadata)
assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

print("✅ Split sin muestras repetidas ni perdidas.")

IDs totales tras el split: 220000
IDs originales: 220000
✅ Split sin muestras repetidas ni perdidas.


In [21]:
print("TRAIN")
display(
    pd.crosstab(
        train_df["modulation"],
        train_df["snr"]
    )
)

print("VALIDATION")
display(
    pd.crosstab(
        validation_df["modulation"],
        validation_df["snr"]
    )
)

print("TEST")
display(
    pd.crosstab(
        test_df["modulation"],
        test_df["snr"]
    )
)

TRAIN


snr,-20,-18,-16,-14,-12,-10,-8,-6,-4,-2,0,2,4,6,8,10,12,14,16,18
modulation,,,,,,,,,,,,,,,,,,,,
8PSK,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700
AM-DSB,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700
AM-SSB,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700
BPSK,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700
CPFSK,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700
GFSK,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700
PAM4,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700
QAM16,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700
QAM64,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700,700


VALIDATION


snr,-20,-18,-16,-14,-12,-10,-8,-6,-4,-2,0,2,4,6,8,10,12,14,16,18
modulation,,,,,,,,,,,,,,,,,,,,
8PSK,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
AM-DSB,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
AM-SSB,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
BPSK,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
CPFSK,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
GFSK,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
PAM4,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
QAM16,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
QAM64,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150


TEST


snr,-20,-18,-16,-14,-12,-10,-8,-6,-4,-2,0,2,4,6,8,10,12,14,16,18
modulation,,,,,,,,,,,,,,,,,,,,
8PSK,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
AM-DSB,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
AM-SSB,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
BPSK,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
CPFSK,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
GFSK,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
PAM4,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
QAM16,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150
QAM64,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150,150


In [23]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_clean = train_df.drop(columns="stratum", errors="ignore")
validation_clean = validation_df.drop(columns="stratum", errors="ignore")
test_clean = test_df.drop(columns="stratum", errors="ignore")

train_clean.to_csv(
    PROCESSED_DIR / "train_metadata.csv",
    index=False
)

validation_clean.to_csv(
    PROCESSED_DIR / "validation_metadata.csv",
    index=False
)

test_clean.to_csv(
    PROCESSED_DIR / "test_metadata.csv",
    index=False
)

print("Train:", train_clean.shape)
print("Validation:", validation_clean.shape)
print("Test:", test_clean.shape)

print("\n✅ Splits guardados en data/processed/")

Train: (154000, 4)
Validation: (33000, 4)
Test: (33000, 4)

✅ Splits guardados en data/processed/


In [24]:
print((PROCESSED_DIR / "train_metadata.csv").exists())
print((PROCESSED_DIR / "validation_metadata.csv").exists())
print((PROCESSED_DIR / "test_metadata.csv").exists())

True
True
True
